In [12]:
from langchain_community.embeddings import HuggingFaceEmbeddings

from milvus_vector import MilvusVectorSave

model_name = "BAAI/bge-small-zh-v1.5"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True}  # set True to compute cosine similarity
embedding_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
    cache_folder="./cache"
)
mv = MilvusVectorSave(embedding_model)
mv.create_connection()

Loading weights: 100%|██████████| 71/71 [00:00<00:00, 5851.29it/s]
BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
# 根据内容检索
mv.vector_store_saved.similarity_search(
    query='现在，最先进的纳米清洗技术是什么?',
    k=2
)

[Document(metadata={'filetype': 'text/markdown', 'category_depth': 1, 'source': './data/md\\tech_report_82gi97rg.md', 'filename': 'tech_report_82gi97rg.md', 'title': '定义与技术背景', 'category': 'content', 'id': 465532897672841410}, page_content='半导体制造中湿法清洗与干法清洗工艺的选择指南 -> 定义与技术背景 湿法清洗(Wet Cleaning)是指利用液体化学试剂（如SC1、SC2溶液）和去离子水，通过化学反应或物理溶解作用去除晶圆表面污染物的工艺。该方法起源于1960年代RCA清洗体系，至今仍是去除颗粒、有机残留和金属污染的主流技术。 干法清洗(Dry Cleaning)则是通过气态或等离子体活性物质（如O₂、CF₄、NF₃）与表面污染物发生反应，生成挥发性产物达到清洁目的的技术。典型代表包括等离子清洗、气相清洗和超临界CO₂清洗，1980年代后随着对纳米级清洁需求而发展。'),
 Document(metadata={'filetype': 'text/markdown', 'category_depth': 2, 'source': './data/md\\tech_report_okefq93j.md', 'filename': 'tech_report_okefq93j.md', 'title': '无残留与高清洁效率', 'category': 'content', 'id': 465532897672842671}, page_content='半导体清洗工艺中超临界流体技术的优势分析 -> 超临界流体清洗的技术优势 -> 无残留与高清洁效率 超临界CO₂的低表面张力特性使其能够渗透到纳米级器件结构的深宽比大于50:1的沟槽中，彻底去除光刻胶（Photoresist）、蚀刻残留物和金属污染物。相比异丙醇（IPA）等传统溶剂，其清洗后的表面颗粒残留可降低90%以上，特别适用于FinFET和GAA(Gate-All-Around)等先进制程。')]

In [4]:
# 根据标量检索
mv.vector_store_saved.similarity_search_with_score(
    query='现在，最先进的纳米清洗技术是什么?',
    k=2,
    expr='category == "Title"'
)

[(Document(metadata={'source': './data/md\\tech_report_okefq93j.md', 'filename': 'tech_report_okefq93j.md', 'filetype': 'text/markdown', 'category_depth': 1, 'category': 'Title', 'title': '超临界流体清洗的技术优势', 'id': 465532897672842670}, page_content='半导体清洗工艺中超临界流体技术的优势分析 -> 超临界流体清洗的技术优势'),
  0.9070048332214355),
 (Document(metadata={'source': './data/md\\tech_report_82gi97rg.md', 'filename': 'tech_report_82gi97rg.md', 'filetype': 'text/markdown', 'category_depth': 0, 'category': 'Title', 'title': '半导体制造中湿法清洗与干法清洗工艺的选择指南', 'id': 465532897672841409}, page_content='半导体制造中湿法清洗与干法清洗工艺的选择指南'),
  0.9041431546211243)]

In [5]:
from pymilvus import MilvusClient
from milvus_vector import MILVUS_URI

# 原生检索
client = MilvusClient(uri=MILVUS_URI)


In [10]:
# 原生检索
client.search(
    collection_name='t_collection01',
    data=['半导体表面特征改善'],
    limit=3,
    anns_field="sparse",  # 使用稀疏索引进行检索 即 使用全文索引BM25
    output_fields=['text', 'id', 'category'],
    search_params={"params": {"drop_ratio_search": 0.2}}  # 忽略低重要性词语比例
)


data: [[{'id': 465532897672840827, 'distance': 10.849193572998047, 'entity': {'text': '等离子体处理改善半导体材料表面特性的技术与方法 -> 半导体表面特性改善的具体应用方向', 'category': 'Title', 'id': 465532897672840827}}, {'id': 465532897672840839, 'distance': 10.510190963745117, 'entity': {'text': '等离子体处理改善半导体材料表面特性的技术与方法 -> 表征技术与效果评估 -> 表面化学分析 X射线光电子能谱(XPS, X-ray Photoelectron Spectroscopy)：定量分析元素价态变化（检测限0.1 at%） 傅里叶变换红外光谱(FTIR, Fourier Transform Infrared)：识别表面官能团（如Si-H在2080 cm⁻¹的特征峰） 飞行时间二次离子质谱(ToF-SIMS, Time-of-Flight SIMS)：ppb级杂质检测与深度剖析', 'category': 'content', 'id': 465532897672840839}}, {'id': 465532897672840834, 'distance': 9.678722381591797, 'entity': {'text': '等离子体处理改善半导体材料表面特性的技术与方法 -> 先进等离子体表面处理技术', 'category': 'Title', 'id': 465532897672840834}}]]

In [16]:
from pymilvus import AnnSearchRequest, RRFRanker

# 混合检索
# 密集向量搜索
req_1 = AnnSearchRequest(
    data=[embedding_model.embed_query('现在，最先进的纳米级清洗技术是什么?')],
    anns_field='dense',
    param={
        "metric_type": "IP",
        "params": {"nprobe": 10}
    },
    limit=5
)
# 稀疏向量搜索
req_2 = AnnSearchRequest(
    data=['半导体表面特征改善'],
    anns_field='sparse',
    param={
        "metric_type": "BM25",
    },
    limit=5
)

client.hybrid_search(
    collection_name='t_collection01',
    reqs=[req_1, req_2],  # 混合检索条件
    ranker=RRFRanker(k=60),
    limit=5,
    output_fields=['text', 'title', 'category'],
)

data: [[{'id': 465532897672840827, 'distance': 0.016393441706895828, 'entity': {'text': '等离子体处理改善半导体材料表面特性的技术与方法 -> 半导体表面特性改善的具体应用方向', 'title': '半导体表面特性改善的具体应用方向', 'category': 'Title'}}, {'id': 465532897672842009, 'distance': 0.016393441706895828, 'entity': {'text': '半导体制造中的纳米级污染物清洁工艺 -> 主流清洁技术分类与原理 -> 干法清洗（Dry Cleaning） 适用于对水分敏感的工艺环节： 等离子体清洗（Plasma Cleaning） 使用O₂/H₂等气体产生活性自由基，分解有机物 可调节离子轰击能量（通常<50eV）避免表面损伤 超临界CO₂清洗 在31.1℃、7.38MPa临界点以上操作 具有气体渗透性和液体溶解力的双重特性 特别适合高深宽比结构（如3D NAND的通道孔清洁）', 'title': '干法清洗（Dry Cleaning）', 'category': 'content'}}, {'id': 465532897672840839, 'distance': 0.016129031777381897, 'entity': {'text': '等离子体处理改善半导体材料表面特性的技术与方法 -> 表征技术与效果评估 -> 表面化学分析 X射线光电子能谱(XPS, X-ray Photoelectron Spectroscopy)：定量分析元素价态变化（检测限0.1 at%） 傅里叶变换红外光谱(FTIR, Fourier Transform Infrared)：识别表面官能团（如Si-H在2080 cm⁻¹的特征峰） 飞行时间二次离子质谱(ToF-SIMS, Time-of-Flight SIMS)：ppb级杂质检测与深度剖析', 'title': '表面化学分析', 'category': 'content'}}, {'id': 465532897672842012, 'distance': 0.016129031777381897, 'entity': {'text': 

In [18]:
# 使用 langchain进行混合检索
mv.vector_store_saved.similarity_search_with_score(
    query='现在，最先进的纳米级清洗技术是什么?',
    k=3,
    ranker_type='rrf',  # rrf/weighted
    ranker_params={'k': 60}
)

[(Document(metadata={'category': 'content', 'title': '气溶胶喷射清洗（Aerosol Jet Cleaning）', 'category_depth': 2, 'source': './data/md\\tech_report_froo52ed.md', 'filename': 'tech_report_froo52ed.md', 'filetype': 'text/markdown', 'id': 465532897672842012}, page_content='半导体制造中的纳米级污染物清洁工艺 -> 面向先进节点的创新工艺 -> 气溶胶喷射清洗（Aerosol Jet Cleaning） 将Ar/N₂混合气体与去离子水雾化成100-300nm液滴 通过文丘里效应加速至超音速 动能控制精度达0.1eV/atom级，可清除10nm以下颗粒'),
  0.0317540317773819),
 (Document(metadata={'category': 'content', 'title': '干法清洗（Dry Cleaning）', 'category_depth': 2, 'source': './data/md\\tech_report_froo52ed.md', 'filename': 'tech_report_froo52ed.md', 'filetype': 'text/markdown', 'id': 465532897672842009}, page_content='半导体制造中的纳米级污染物清洁工艺 -> 主流清洁技术分类与原理 -> 干法清洗（Dry Cleaning） 适用于对水分敏感的工艺环节： 等离子体清洗（Plasma Cleaning） 使用O₂/H₂等气体产生活性自由基，分解有机物 可调节离子轰击能量（通常<50eV）避免表面损伤 超临界CO₂清洗 在31.1℃、7.38MPa临界点以上操作 具有气体渗透性和液体溶解力的双重特性 特别适合高深宽比结构（如3D NAND的通道孔清洁）'),
  0.016393441706895828),
 (Document(metadata={'category': 'content', 'title': '光刻胶图形化（Photore

In [25]:
# 使用langchain链式检索
vector_store = mv.vector_store_saved.as_retriever(
    search_type='similarity',  # 只返回相似度大于阈值的结果
    search_kwargs={
        'k': 3,
        'ranker_type': 'rrf',
        'ranker_params': {'k': 100},
        'score_threshold': 0.1,
        'filter':{'category':'Content'}
    }
)
vector_store.invoke("介绍一下: 光刻机有哪几种?")

[Document(metadata={'title': '材料创新方向', 'category': 'content', 'source': './data/md\\tech_report_epzry4lv.md', 'filename': 'tech_report_epzry4lv.md', 'filetype': 'text/markdown', 'category_depth': 2, 'id': 465532897672841899}, page_content='光刻胶分辨率极限及其突破方法 -> 突破分辨率限制的技术路径 -> 材料创新方向 金属氧化物光刻胶(Metal Oxide Resist)： 采用氧化锡、氧化铪等无机-有机杂化材料 优势：高EUV吸收率、低LER(可达1.2nm)、抗刻蚀性强 挑战：灵敏度偏低(通常>30mJ/cm²) 分子玻璃光刻胶(Molecular Glass Resist)： 由单分散性有机分子构成 可精确控制分子尺寸(1-2nm)和形状 典型代表：杯芳烃(Calixarene)衍生物 二嵌段共聚物(DSA, Directed Self-Assembly)： 利用PS-b-PMMA等嵌段共聚物的相分离特性 结合预图案导向技术，可实现5nm以下特征尺寸 需要精确控制退火温度和界面能'),
 Document(metadata={'title': '光刻机数值孔径的定义与物理意义', 'category': 'content', 'source': './data/md\\tech_report_tpsi5pi8.md', 'filename': 'tech_report_tpsi5pi8.md', 'filetype': 'text/markdown', 'category_depth': 1, 'id': 465532897672843085}, page_content='光刻机的数值孔径及其对分辨率的影响 -> 光刻机数值孔径的定义与物理意义 数值孔径（Numerical Aperture, NA）是光学光刻系统中最重要的参数之一，定义为物镜折射率(n)与入射光半角(θ)正弦值的乘积(NA = n×sinθ)。在半导体制造中，光刻机数值孔径直接影响图案转移的精度，其物理意义表征了光学系统收集衍射光的能力。典型的深紫外